# Credit Policy Lab (simple version)

**One sentence:** use past loans to predict who defaults, turn that prediction into expected profit per loan, approve only loans that are expected to make money, and check the result on a later year the model never saw.

**How the notebook flows (each step is one code cell, with a short note above it):**
1. Get the data  2. Pick the loans  3. Build features  4. Train two models  5. Make probabilities trustworthy  6. Compare models  7. Convert to money  8. Choose the approval rule  9. Stress test  10. Write the memo

**Rules I follow so the result is honest:**
- Train on 2012-2013, tune on 2014, test once on 2015 (split by date, never random).
- Models never see `int_rate`, `grade` or `installment` (the lender's own risk price).
- No accuracy metric: bad loans are a minority, so "accuracy" would be misleading.
- Every choice (model settings, approval cutoff) is made on 2014 data, not on the test year.

## 1. Setup and data

**What it does:** imports libraries, sets the assumptions, and finds the data file. If the file is not in `credit_policy_lab/data/`, it downloads it from Kaggle once.

**Assumptions to know:** cost of funds 5% a year, average balance outstanding 1.65 years (3 years x 0.55), running cost 1.5% of the loan. They are assumptions, not facts; the stress test shows what happens if they are wrong.

In [ ]:
DEMO = False   # True = use a 150k-loan sample for a quick test run
SEED = 42

import sys, os, json, shutil, subprocess, importlib, warnings
from pathlib import Path


def ensure(pkg, module):
    """Import a module, installing the package first if it is missing."""
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return importlib.import_module(module)


for pkg, mod in [("pandas", "pandas"), ("numpy", "numpy"), ("scikit-learn", "sklearn"), ("lightgbm", "lightgbm"),
                 ("matplotlib", "matplotlib"), ("pyarrow", "pyarrow")]:
    ensure(pkg, mod)

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/credit_policy_lab")
else:
    BASE = Path("credit_policy_lab")
DATA_DIR, REPORT_DIR = BASE / "data", BASE / "reports"
for d in (DATA_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# economic assumptions
COF = 0.05          # annual cost of funds
AVG_LIFE = 1.65     # years of average balance outstanding (3 years x 0.55)
OPEX = 0.015        # running cost as a share of the funded amount
TERM = 36           # loan term in months kept in the study
TRAIN_END, VALID_END, TEST_END = "2013-12-01", "2014-12-01", "2015-12-01"
START = "2012-01-01"


def find_data():
    """Use the env override, else a file in data/, else download from Kaggle into data/."""
    if os.environ.get("CPL_DATA_FILE"):
        return Path(os.environ["CPL_DATA_FILE"])
    def ok(p):
        n = p.name.lower()
        return p.is_file() and n.startswith("accepted_2007_to_2018q4") and n.endswith((".csv.gz", ".csv"))
    found = [p for p in DATA_DIR.glob("*") if ok(p)]
    if not found:
        kagglehub = ensure("kagglehub", "kagglehub")
        path = Path(kagglehub.dataset_download("wordsforthewise/lending-club"))
        src = sorted((p for p in path.rglob("*") if ok(p)), key=lambda p: not p.name.endswith(".gz"))
        if not src:
            raise FileNotFoundError(f"accepted_2007_to_2018Q4 file not found under {path}")
        shutil.copy2(src[0], DATA_DIR / src[0].name)
        found = [DATA_DIR / src[0].name]
    return found[0]


DATA_FILE = find_data()
print("data file:", DATA_FILE)

## 2. Load and pick the loans

**What it does:**
- Reads only the columns needed.
- Keeps 36-month loans issued 2012 to 2015.
- Marks a loan `bad = 1` if Charged Off or Default, `0` if Fully Paid.
- **Maturity filter:** drops issue months where more than 1% of loans are still running. Reason: loans that finish early are mostly good ones, so recent months with many running loans would look safer than they really are.
- Splits by date: train 2012-2013, validation 2014, test 2015.

**What to look at in the output:** the three rows (rows, bad rate per split) and the chart. Test bad rate usually differs from train: that is drift, and it is deliberate.

In [ ]:
COLS = ["id", "issue_d", "earliest_cr_line", "loan_status", "term", "int_rate", "installment", "grade",
        "funded_amnt", "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
        "collection_recovery_fee", "loan_amnt", "emp_length", "home_ownership", "annual_inc", "verification_status",
        "purpose", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util",
        "total_acc", "fico_range_low", "fico_range_high"]

raw = pd.read_csv(DATA_FILE, usecols=COLS, low_memory=False)
raw["issue_d"] = pd.to_datetime(raw["issue_d"], format="%b-%Y", errors="coerce")
raw["earliest_cr_line"] = pd.to_datetime(raw["earliest_cr_line"], format="%b-%Y", errors="coerce")
raw = raw[raw["issue_d"].notna()]                        # the file ends with summary rows that have no date
raw["term_m"] = pd.to_numeric(raw["term"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
raw = raw[(raw["term_m"] == TERM) & (raw["issue_d"] >= START) & (raw["issue_d"] <= TEST_END)].copy()
if DEMO:
    raw = raw.sample(n=min(150_000, len(raw)), random_state=SEED)

# maturity filter
unresolved = raw["loan_status"].isin(["Current", "In Grace Period"]) | raw["loan_status"].astype(str).str.startswith("Late")
share = unresolved.groupby(raw["issue_d"]).mean()
keep_months = share[share <= 0.01].index
print(f"maturity filter: kept {len(keep_months)} of {len(share)} issue months")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(share.index, share.values)
ax.axhline(0.01, color="grey", ls="--")
ax.set(title="Share of loans still unresolved, by issue month", ylabel="unresolved share")
plt.show()

df = raw[raw["issue_d"].isin(keep_months) & raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["bad"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)
df["split"] = np.where(df["issue_d"] <= TRAIN_END, "train", np.where(df["issue_d"] <= VALID_END, "valid", "test"))
df = df.sort_values(["issue_d", "id"]).reset_index(drop=True)

summary = df.groupby("split").agg(loans=("bad", "size"), first=("issue_d", "min"), last=("issue_d", "max"),
                                  bad_rate=("bad", "mean")).loc[["train", "valid", "test"]]
display(summary)
assert df[df.split == "train"].issue_d.max() < df[df.split == "valid"].issue_d.min() < df[df.split == "test"].issue_d.min()

## 3. Features

**What it does:** builds simple applicant features known on the day of the application (average FICO, credit history length, log income, loan-to-income, DTI, revolving utilisation, employment years, a few counts, and home / verification / purpose as dummy columns). Missing values get the **train** median.

**Leakage check:** the code stops if any banned column (`int_rate`, `grade`, `installment`, anything observed after origination) is in the feature list. `installment` is banned because at a fixed 36-month term, installment divided by loan amount is just a function of `int_rate`.

In [ ]:
BANNED = {"int_rate", "grade", "sub_grade", "installment", "loan_status", "bad", "funded_amnt", "total_pymnt",
          "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee"}

f = pd.DataFrame(index=df.index)
f["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
f["credit_history_years"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25
f["log_income"] = np.log1p(df["annual_inc"])
f["loan_amnt"] = df["loan_amnt"]
f["loan_to_income"] = (df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)).clip(upper=2.0)
f["dti"] = df["dti"].clip(0, 100)
f["revol_util"] = df["revol_util"].clip(0, 150)
f["log_revol_bal"] = np.log1p(df["revol_bal"])
f["open_acc_ratio"] = df["open_acc"] / df["total_acc"].replace(0, np.nan)
f["delinq_2yrs"] = df["delinq_2yrs"].clip(upper=5)
f["inq_last_6mths"] = df["inq_last_6mths"].clip(upper=6)
f["pub_rec"] = df["pub_rec"].clip(upper=3)
emp = df["emp_length"].astype("string")
f["emp_years"] = pd.to_numeric(emp.str.extract(r"(\d+)")[0], errors="coerce").where(~emp.str.contains("<", na=False), 0.0)
f["emp_missing"] = f["emp_years"].isna().astype(int)
dummies = pd.get_dummies(df[["home_ownership", "verification_status", "purpose"]].astype(str), dtype=float)
X = pd.concat([f, dummies], axis=1)

is_train = (df["split"] == "train").to_numpy()
X = X.fillna(X[is_train].median())                       # train-only medians
FEATURES = list(X.columns)
assert not (set(FEATURES) & BANNED), "banned column in features"
assert not X.isna().any().any()

Xs = {s: X[(df["split"] == s).to_numpy()] for s in ["train", "valid", "test"]}
D = {s: df[df["split"] == s].reset_index(drop=True) for s in ["train", "valid", "test"]}
for s in Xs:
    Xs[s] = Xs[s].reset_index(drop=True)
print(f"{len(FEATURES)} features; leakage check passed (no int_rate, grade or installment)")

bands = pd.cut(f["fico_avg"][is_train], [0, 680, 700, 720, 740, 760, 900], right=False)
rate = df["bad"][is_train].groupby(bands).mean()
rate.plot(kind="bar", figsize=(6, 3), title="Train bad rate by FICO band", ylabel="bad rate")
plt.show()

## 4. Two models

- **Logistic regression** (the "scorecard-style" model): simple, transparent, one weight per feature. Regularisation strength `C` is chosen on **2014** data.
- **LightGBM** (the challenger): many small decision trees, with early stopping on 2014. Monotone constraints force sensible directions (higher FICO can only lower risk; higher DTI can only raise it).

**What to look at:** which `C` won, and how many trees LightGBM kept. Nothing is tuned on 2015.

In [ ]:
ytr, yva, yte = D["train"]["bad"], D["valid"]["bad"], D["test"]["bad"]

scaler = StandardScaler().fit(Xs["train"])
best = None
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    m = LogisticRegression(C=C, max_iter=2000).fit(scaler.transform(Xs["train"]), ytr)
    auc = roc_auc_score(yva, m.predict_proba(scaler.transform(Xs["valid"]))[:, 1])
    print(f"[valid] C={C}: AUC {auc:.4f}")
    if best is None or auc > best[0]:
        best = (auc, C, m)
logit_model = best[2]
print("chosen C:", best[1])

MONO = {"fico_avg": -1, "log_income": -1, "dti": 1, "revol_util": 1, "inq_last_6mths": 1,
        "delinq_2yrs": 1, "pub_rec": 1, "loan_to_income": 1}
params = {"objective": "binary", "metric": "auc", "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 200,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10.0, "seed": SEED,
          "deterministic": True, "force_row_wise": True, "verbose": -1,
          "monotone_constraints": [MONO.get(c, 0) for c in FEATURES]}
booster = lgb.train(params, lgb.Dataset(Xs["train"], ytr), num_boost_round=150 if DEMO else 3000,
                    valid_sets=[lgb.Dataset(Xs["valid"], yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
print("LightGBM trees kept:", booster.best_iteration)

raw_pd = {s: {"logit": logit_model.predict_proba(scaler.transform(Xs[s]))[:, 1],
              "lgbm": booster.predict(Xs[s], num_iteration=booster.best_iteration)} for s in Xs}

imp = pd.Series(booster.feature_importance("gain"), index=FEATURES).sort_values().tail(10)
imp.plot(kind="barh", figsize=(6, 3.5), title="LightGBM: top features by gain")
plt.show()

## 5. Calibration

**What it does:** a model can rank loans well but still say "10% risk" when the real rate is 14%. Profit = probability x dollars, so wrong probabilities mean wrong profit. I fit **isotonic regression** on 2014 predictions to correct this.

**What to look at:** the decile table on the 2015 test set. Mean predicted PD and actual bad rate should be close in each row. If test predictions are a bit low overall, that is drift: 2015 was riskier than 2014 and no calibration can see the future. Brier score should drop slightly after calibration.

In [ ]:
iso = {k: IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(raw_pd["valid"][k], yva) for k in ["logit", "lgbm"]}
cal_pd = {s: {k: iso[k].predict(raw_pd[s][k]) for k in iso} for s in Xs}

dec = pd.qcut(pd.Series(cal_pd["test"]["lgbm"]).rank(method="first"), 10, labels=False) + 1
calib = pd.DataFrame({"mean_predicted_pd": pd.Series(cal_pd["test"]["lgbm"]).groupby(dec).mean(),
                      "actual_bad_rate": yte.groupby(dec).mean()})
print("[test] LightGBM calibration by decile (decile 10 = riskiest)")
display(calib.round(4))
for k in iso:
    print(f"[test] Brier {k}: raw {brier_score_loss(yte, raw_pd['test'][k]):.5f} -> calibrated {brier_score_loss(yte, cal_pd['test'][k]):.5f}")
print(f"[test] mean calibrated PD (lgbm) {cal_pd['test']['lgbm'].mean():.4f} vs actual bad rate {yte.mean():.4f}")

## 6. Compare models (out of time)

**What it does:** reports AUC, KS and Gini for train, 2014 and 2015, for both models and for the lender's own `int_rate` used as a score.

- **AUC:** chance a random bad loan gets a riskier score than a random good one (0.5 = coin flip, 1 = perfect).
- **KS:** biggest gap between how bads and goods are spread across scores.
- **Gini** = 2 x AUC - 1.

**What to look at:** train AUC should be higher than test (some overfit is normal). The `int_rate` row is the benchmark: the lender priced with information I do not have, so it is hard to beat. If a model's test AUC is above 0.80, the code prints a leakage warning.

In [ ]:
def ks(y, s):
    """Largest gap between cumulative bad and good distributions."""
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr - fpr))


rows = []
for s in ["train", "valid", "test"]:
    y = D[s]["bad"]
    for name, score in [("logistic", raw_pd[s]["logit"]), ("lightgbm", raw_pd[s]["lgbm"]), ("int_rate baseline", D[s]["int_rate"])]:
        auc = roc_auc_score(y, score)
        rows.append({"split": s, "model": name, "AUC": auc, "KS": ks(y, score), "Gini": 2 * auc - 1})
metrics = pd.DataFrame(rows).set_index(["split", "model"])
print("settings were chosen on valid; test was used only to report")
display(metrics.round(4))

test_auc = metrics.loc["test"]["AUC"]
for name in ["logistic", "lightgbm"]:
    if test_auc[name] > 0.80:
        print(f"WARNING: {name} test AUC {test_auc[name]:.3f} is above 0.80; this almost certainly means leakage.")

fig, ax = plt.subplots(figsize=(5, 4.5))
for name, score in [("logistic", raw_pd["test"]["logit"]), ("lightgbm", raw_pd["test"]["lgbm"]), ("int_rate", D["test"]["int_rate"])]:
    fpr, tpr, _ = roc_curve(yte, score)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(yte, score):.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.7)
ax.set(title="ROC on test (2015)", xlabel="goods flagged", ylabel="bads caught")
ax.legend()
plt.show()

## 7. Convert risk into money

**Expected profit of one loan:**

`EV = funded x [ (1 - PD) x realization x contractual_yield + PD x bad_result ] - funded x COF x 1.65 - funded x opex`

- `realization`: share of promised interest actually collected on good loans (below 1 because people repay early). Measured on **train**.
- `bad_result`: net cash per dollar funded on bad loans (negative; partial repayments and recoveries are already in it). Measured on **train**.

**Realised profit (for the backtest) uses real cash flows:** interest income = interest + late fees; credit loss = interest income minus (cash received - collection fees - amount funded); net profit = interest income - credit loss - funding cost - running cost.

**What to look at:** the match rate should be close to 100%. The risk-band table shows which score bands lose money in reality.

In [ ]:
for s in D:
    d = D[s]
    d["cont_yield"] = (d["installment"] * TERM - d["funded_amnt"]) / d["funded_amnt"]
    d["interest_income"] = d["total_rec_int"] + d["total_rec_late_fee"]
    gross = d["total_pymnt"] - d["collection_recovery_fee"] - d["funded_amnt"]
    d["credit_loss"] = d["interest_income"] - gross
    d["cost"] = d["funded_amnt"] * (COF * AVG_LIFE + OPEX)
    d["net_profit"] = d["interest_income"] - d["credit_loss"] - d["cost"]
    d["pd_logit"], d["pd_lgbm"] = cal_pd[s]["logit"], cal_pd[s]["lgbm"]

allc = pd.concat(D.values())
match = ((allc["total_pymnt"] - (allc["total_rec_prncp"] + allc["total_rec_int"] + allc["total_rec_late_fee"] + allc["recoveries"])).abs() <= 1)
print(f"total_pymnt = principal + interest + late fees + recoveries (within $1) for {match.mean():.2%} of loans")

tr = D["train"]
good, bad = tr[tr.bad == 0], tr[tr.bad == 1]
REALIZATION = good["total_rec_int"].sum() / (good["installment"] * TERM - good["funded_amnt"]).sum()
BAD_RESULT = (bad["total_pymnt"] - bad["collection_recovery_fee"] - bad["funded_amnt"]).sum() / bad["funded_amnt"].sum()
print(f"[train] realization {REALIZATION:.3f} | bad_result {BAD_RESULT:.3f}")


def expected_value(d, pdv, cof=COF, pd_mult=1.0):
    """Expected profit per loan in dollars."""
    p = np.clip(np.asarray(pdv) * pd_mult, 0, 1)
    fnd = d["funded_amnt"].to_numpy()
    return fnd * ((1 - p) * REALIZATION * d["cont_yield"].to_numpy() + p * BAD_RESULT) - fnd * (cof * AVG_LIFE + OPEX)


for s in ["valid", "test"]:
    D[s]["ev_logit"] = expected_value(D[s], D[s]["pd_logit"])
    D[s]["ev_lgbm"] = expected_value(D[s], D[s]["pd_lgbm"])

# realised P&L by score band (band edges from valid)
te = D["test"]
edges = np.quantile(D["valid"]["pd_lgbm"], np.linspace(0, 1, 7)[1:-1])
te["band"] = 6 - np.searchsorted(edges, te["pd_lgbm"], side="right")      # band 1 = riskiest
band_tbl = te.groupby("band").agg(loans=("bad", "size"), predicted_pd=("pd_lgbm", "mean"), actual_bad_rate=("bad", "mean"),
                                  avg_int_rate=("int_rate", "mean"), interest_income=("interest_income", "sum"),
                                  credit_loss=("credit_loss", "sum"), net_profit=("net_profit", "sum"), funded=("funded_amnt", "sum"))
band_tbl["roi"] = band_tbl["net_profit"] / band_tbl["funded"]
for c in ["interest_income", "credit_loss", "net_profit"]:
    band_tbl[c] = band_tbl[c] / 1e6
print("[test] realised P&L by risk band, $ in millions (band 1 = riskiest)")
display(band_tbl.drop(columns="funded").round(4))

## 8. Choose the approval rule and backtest it

**Rule:** approve a loan when its expected profit is above a **hurdle**. It is not one PD cutoff because interest differs by loan: a riskier loan with a high rate can be worth more than a safe loan with a low rate.

**Policies compared:** approve all; the lender's own grade policy (approve grades up to a cutoff); logistic EV policy; LightGBM EV policy.

**How it is chosen without cheating:** hurdle and grade cutoff are picked on **2014** to maximise realised profit; each policy is then run **once on 2015**. The champion is whichever model policy earned more on 2014.

**What to look at:** the 2015 table (approval rate, net profit, ROI, bad rate of approved loans) and the profit bridge. Important limits: the data holds only loans the lender already approved, so the policy can only tighten approvals; and a policy can look worse on test than on validation if 2015 was riskier.

In [ ]:
va = D["valid"]


def best_hurdle(ev, net):
    """Hurdle on expected profit that maximises realised net profit when approving loans above it."""
    cands = np.unique(np.r_[ev.min() - 1, 0.0, np.quantile(ev, np.linspace(0, 1, 101))])
    profits = [net[ev > h].sum() for h in cands]
    return float(cands[int(np.argmax(profits))])


H = {k: best_hurdle(va[f"ev_{k}"].to_numpy(), va["net_profit"].to_numpy()) for k in ["logit", "lgbm"]}
grades = sorted(D["train"]["grade"].astype(str).unique())
rank = {g: i for i, g in enumerate(grades)}
grade_cut = max(grades, key=lambda g: va.loc[va["grade"].astype(str).map(rank) <= rank[g], "net_profit"].sum())
print(f"[valid] hurdles: logistic {H['logit']:,.0f}$, lightgbm {H['lgbm']:,.0f}$ | grade policy approves grades {grades[0]} to {grade_cut}")


def masks(d):
    """Approval decision of each policy."""
    return {"approve all": np.ones(len(d), bool),
            "grade policy": (d["grade"].astype(str).map(rank) <= rank[grade_cut]).to_numpy(),
            "logistic EV": (d["ev_logit"] > H["logit"]).to_numpy(),
            "lightgbm EV": (d["ev_lgbm"] > H["lgbm"]).to_numpy()}


def pnl(d, m):
    """Portfolio result of the approved loans."""
    a = d[m]
    return {"approval_rate": m.mean(), "funded_$M": a["funded_amnt"].sum() / 1e6, "interest_$M": a["interest_income"].sum() / 1e6,
            "credit_loss_$M": a["credit_loss"].sum() / 1e6, "cost_$M": a["cost"].sum() / 1e6,
            "net_profit_$M": a["net_profit"].sum() / 1e6, "roi": a["net_profit"].sum() / max(a["funded_amnt"].sum(), 1),
            "bad_rate": a["bad"].mean()}


valid_tbl = pd.DataFrame({k: pnl(va, m) for k, m in masks(va).items()}).T
print("[valid] used to choose the champion")
display(valid_tbl[["approval_rate", "net_profit_$M", "roi"]].round(4))
CHAMP = "lightgbm EV" if valid_tbl.loc["lightgbm EV", "net_profit_$M"] >= valid_tbl.loc["logistic EV", "net_profit_$M"] else "logistic EV"
CHAMP_KEY = "lgbm" if CHAMP == "lightgbm EV" else "logit"
print("champion chosen on valid:", CHAMP)

test_masks = masks(te)
test_tbl = pd.DataFrame({k: pnl(te, m) for k, m in test_masks.items()}).T
print("[test] each policy evaluated once, hurdles fixed on valid")
display(test_tbl.round(4))

base, pol = test_tbl.loc["approve all"], test_tbl.loc[CHAMP]
steps = {"interest forgone": -(base["interest_$M"] - pol["interest_$M"]),
         "credit losses avoided": base["credit_loss_$M"] - pol["credit_loss_$M"],
         "funding+running cost saved": base["cost_$M"] - pol["cost_$M"]}
assert abs(sum(steps.values()) - (pol["net_profit_$M"] - base["net_profit_$M"])) < 1e-6, "bridge does not reconcile"
level = base["net_profit_$M"]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(0, level, color="grey")
for i, (k, v) in enumerate(steps.items(), 1):
    ax.bar(i, v, bottom=level, color="tab:green" if v >= 0 else "tab:red")
    level += v
ax.bar(4, pol["net_profit_$M"], color="tab:blue")
ax.set_xticks(range(5), ["approve all", *steps, CHAMP], rotation=15)
ax.axhline(0, color="k", lw=0.6)
ax.set(title="Test net profit bridge ($M)", ylabel="$ millions")
plt.show()

## 9. Stress test (model-based, not realised)

**What it does:** keeps the champion's approved loans fixed and recomputes **expected** profit if funding gets more expensive and defaults are higher than predicted. It also finds the breakeven PD multiplier: how much worse defaults can get before the approved book stops making money.

**What to look at:** the profit grid (rows = cost of funds, columns = PD multiplier), whether the policy still beats approve-all in every cell, and the breakeven multiplier. A breakeven near 1 means no cushion.

In [ ]:
approved = test_masks[CHAMP]
pdv = te[f"pd_{CHAMP_KEY}"].to_numpy()
rows = []
for add in [0.0, 0.02, 0.04]:
    for m in [1.0, 1.25, 1.5]:
        ev = expected_value(te, pdv, cof=COF + add, pd_mult=m)
        rows.append({"cost_of_funds": COF + add, "pd_multiplier": m, "policy_$M": ev[approved].sum() / 1e6,
                     "approve_all_$M": ev.sum() / 1e6})
grid = pd.DataFrame(rows)
grid["policy_beats_all"] = grid["policy_$M"] > grid["approve_all_$M"]
print("[test, model-based] expected profit under stress")
display(grid.round(3))
print(grid.pivot(index="cost_of_funds", columns="pd_multiplier", values="policy_$M").round(2))


def breakeven(cof):
    """PD multiplier where expected profit of the approved loans hits zero (bisection)."""
    f = lambda m: expected_value(te, pdv, cof=cof, pd_mult=m)[approved].sum()
    lo, hi = 0.0, 10.0
    if f(lo) <= 0:
        return 0.0
    if f(hi) > 0:
        return float("inf")
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) > 0 else (lo, mid)
    return (lo + hi) / 2


BREAKEVEN = breakeven(COF)
print("breakeven PD multiplier at base funding cost:", "not reached" if np.isinf(BREAKEVEN) else f"x{BREAKEVEN:.2f}")

## 10. Credit committee memo and results file

**What it does:** writes a plain-language memo from the computed variables (no number is typed by hand) and saves `reports/results_summary.json`. **This memo is your answer to "what is the result?"** Read it top to bottom: recommendation, profit versus approve-all and versus the lender's grade policy, drift, stress result, risks.

In [ ]:
def m(x):
    """Dollar amount in millions."""
    return f"{'-' if x < 0 else ''}${abs(x):,.2f}M"


worst = grid[(grid.cost_of_funds == grid.cost_of_funds.max()) & (grid.pd_multiplier == grid.pd_multiplier.max())].iloc[0]
drift_gap = te["bad"].mean() / D["train"]["bad"].mean() - 1
champ_auc = test_auc["lightgbm" if CHAMP_KEY == "lgbm" else "logistic"]
mean_pd = te[f"pd_{CHAMP_KEY}"].mean()
be_text = "is not reached in the tested range" if np.isinf(BREAKEVEN) else f"is about x{BREAKEVEN:.2f}"
beats_grade = pol["net_profit_$M"] > test_tbl.loc["grade policy", "net_profit_$M"]

memo = f"""
**Recommendation.** Use the {CHAMP} policy: approve a loan when its expected profit is above {H[CHAMP_KEY]:,.0f} dollars, a cutoff chosen on the validation year.
On the untouched test year it approves {pol['approval_rate']:.1%} of loans and earns {m(pol['net_profit_$M'])}, against {m(base['net_profit_$M'])} if every loan were approved, a difference of {m(pol['net_profit_$M'] - base['net_profit_$M'])}.
The lender's own grade policy earns {m(test_tbl.loc['grade policy', 'net_profit_$M'])}, so the model policy {'beats' if beats_grade else 'does not beat'} it by {m(pol['net_profit_$M'] - test_tbl.loc['grade policy', 'net_profit_$M'])}.
Return on funded dollars goes from {base['roi']:.1%} to {pol['roi']:.1%}, and the bad rate of approved loans from {base['bad_rate']:.1%} to {pol['bad_rate']:.1%}.
The model ranks risk with a test AUC of {champ_auc:.3f}, compared with {test_auc['int_rate baseline']:.3f} for the lender's own interest rate.
The bad rate moved from {D['train']['bad'].mean():.1%} in training to {te['bad'].mean():.1%} in test, while the model's mean predicted default rate on test was {mean_pd:.1%}, so {'the model under-predicted in the riskier year' if mean_pd < te['bad'].mean() else 'the model was not too optimistic'}.
Under the harshest stress tested (cost of funds {worst['cost_of_funds']:.1%} and every default probability multiplied by {worst['pd_multiplier']:.2f}) expected profit is {m(worst['policy_$M'])}, and the policy {'still beats' if worst['policy_beats_all'] else 'no longer beats'} approving everyone.
At today's funding cost, the default multiplier at which the approved book breaks even {be_text}.
Main risks: outcomes exist only for loans the original lender approved, so this policy can only tighten approvals; funding and running costs are assumptions.
With a lender's own data I would add rejected applicants, monthly repayment history, and MSME-specific inputs such as invoices and bank statements.
"""
display(Markdown(memo))

summary = {"champion": CHAMP, "hurdle_usd": H[CHAMP_KEY], "grade_cutoff": grade_cut,
           "test_approval_rate": float(pol["approval_rate"]), "test_net_profit_policy_musd": float(pol["net_profit_$M"]),
           "test_net_profit_approve_all_musd": float(base["net_profit_$M"]),
           "test_net_profit_grade_policy_musd": float(test_tbl.loc["grade policy", "net_profit_$M"]),
           "test_roi_policy": float(pol["roi"]), "test_roi_approve_all": float(base["roi"]),
           "test_auc_logistic": float(test_auc["logistic"]), "test_auc_lightgbm": float(test_auc["lightgbm"]),
           "test_auc_int_rate": float(test_auc["int_rate baseline"]), "train_bad_rate": float(D["train"]["bad"].mean()),
           "test_bad_rate": float(te["bad"].mean()), "test_mean_pd": float(mean_pd),
           "realization": float(REALIZATION), "bad_result": float(BAD_RESULT),
           "breakeven_pd_multiplier": None if np.isinf(BREAKEVEN) else float(BREAKEVEN), "demo": DEMO}
(REPORT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=1))
print("saved", REPORT_DIR / "results_summary.json")

## Limitations

- US consumer loans, not MSME credit: the method transfers, the numbers do not.
- Only approved loans have outcomes (reject inference problem), so the policy can only be tightened.
- Profit ignores time value of money; funding cost and running cost are assumptions.
- Final status only, no monthly payment history.
- Expected-value model is simple: one average loss parameter and one prepayment factor.
- Origination data only; no behavioural or bureau-trend data.